In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from config import TOKENIZED_DIR, FEATURE_DIR

# Step 11
class ClipCapDataset(Dataset):
    def __init__(self, tokenized_path, clip_feature_data):
        self.tokenized_data = torch.load(tokenized_path)
        self.image_ids = self.tokenized_data["image_ids"] #list
        self.input_ids = self.tokenized_data["input_ids"] #string
        self.attention_mask = self.tokenized_data["attention_mask"] #string
        
        # matrix các vector feature ảnh
        self.clip_features = clip_feature_data["features"]
        
        # hashmap: img_id (name of img) -> idx
        self.img_id_to_idx = {
            img_id: idx 
            for idx, img_id in enumerate(clip_feature_data["image_ids"])
        }

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        feature_idx = self.img_id_to_idx[img_id]
        image_embed = self.clip_features[feature_idx]
        input_ids = self.input_ids[idx]
        attention_mask = self.attention_mask[idx]

        labels = input_ids.clone()
        labels[attention_mask == 0] = -100 #gán -100 cho các vị trí đệm để hàm CrossEntropyLoss skip
        
        return {
            "image_embed": image_embed,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [2]:
# Step 12
def create_dataloaders(batch_size=32, train_filename="train.pt"):
    clip_feature_path = FEATURE_DIR / "clip_features.pt"
    clip_feature_data = torch.load(clip_feature_path)
    
    dataloaders = {}

    train_path = TOKENIZED_DIR / train_filename
    train_dataset = ClipCapDataset(train_path, clip_feature_data)
    dataloaders["train"] = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    
    val_dataset = ClipCapDataset(TOKENIZED_DIR / "val.pt", clip_feature_data)
    dataloaders["val"] = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    test_dataset = ClipCapDataset(TOKENIZED_DIR / "test.pt", clip_feature_data)
    dataloaders["test"] = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return dataloaders

In [3]:
#Testing
loaders = create_dataloaders(batch_size=32)

print("Check shape của batch 1")
sample_batch = next(iter(loaders["train"]))

print(f"Image Embed Shape  : {sample_batch['image_embed'].shape}")
print(f"Input IDs Shape    : {sample_batch['input_ids'].shape}")
print(f"Attention Mask     : {sample_batch['attention_mask'].shape}")
print(f"Labels Shape       : {sample_batch['labels'].shape}")

print("\n• Sample Labels:")
print(sample_batch['labels'][0])

Check shape của batch 1
Image Embed Shape  : torch.Size([32, 512])
Input IDs Shape    : torch.Size([32, 48])
Attention Mask     : torch.Size([32, 48])
Labels Shape       : torch.Size([32, 48])

• Sample Labels:
tensor([   32,  2415,   319,   257,  8223, 18045,   281, 22007,   764,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100])
